# Stubs and Drivers in Integration Testing

## Introduction

When performing integration testing, we often need to test components before all their dependencies are available. This is where **stubs** and **drivers** come into play.

### Stubs

A **stub** is a simplified implementation of a component that provides canned responses to calls made during testing. Stubs are used in **Top-Down** integration testing where we test higher-level modules before lower-level ones are ready.

### Drivers

A **driver** is a component that calls the module being tested, providing input and capturing output. Drivers are used in **Bottom-Up** integration testing where we test lower-level modules before higher-level ones are ready.

---

## Example: Stub for Storage

Let's create a stub for the Storage class that simulates its behavior without actually storing data.

In [ ]:
class StorageStub:
    """Stub implementation of Storage for Top-Down testing."""
    
    def __init__(self, should_fail=False):
        self.tasks = []
        self.should_fail = should_fail
    
    def save(self, task):
        if self.should_fail:
            raise Exception("Storage failed")
        self.tasks.append(task)
        return True
    
    def get_all(self):
        return self.tasks.copy()

# Test the stub
stub = StorageStub()
stub.save({'title': 'Test Task'})
print(f"Tasks in stub: {stub.get_all()}")

## Example: Stub for Notifier

Now let's create a stub for the Notifier class.

In [ ]:
class NotifierStub:
    """Stub implementation of Notifier for Top-Down testing."""
    
    def __init__(self, should_fail=False):
        self.notifications = []
        self.should_fail = should_fail
    
    def send(self, message):
        if self.should_fail:
            raise Exception("Notifier failed")
        self.notifications.append(message)
        return True
    
    def get_notifications(self):
        return self.notifications.copy()

# Test the stub
notifier_stub = NotifierStub()
notifier_stub.send("Test notification")
print(f"Notifications in stub: {notifier_stub.get_notifications()}")

## Example: Driver for Storage

A driver is used to test a lower-level module (like Storage) in isolation.

In [ ]:
from src.storage import Storage

class StorageDriver:
    """Driver to test Storage module in isolation."""
    
    def __init__(self):
        self.storage = Storage()
    
    def test_valid_task(self):
        """Test saving a valid task."""
        task = {'title': 'Valid Task', 'description': 'Test'}
        result = self.storage.save(task)
        return result
    
    def test_empty_title(self):
        """Test saving a task with empty title."""
        try:
            task = {'title': '', 'description': 'Test'}
            self.storage.save(task)
            return False  # Should have raised an error
        except ValueError:
            return True  # Correctly raised an error
    
    def test_duplicate_task(self):
        """Test saving a duplicate task."""
        task = {'title': 'Duplicate', 'description': 'Test'}
        self.storage.save(task)
        try:
            self.storage.save(task)  # Try to save again
            return False  # Should have raised an error
        except ValueError:
            return True  # Correctly raised an error

# Run driver tests
driver = StorageDriver()
print(f"Valid task test: {driver.test_valid_task()}")
print(f"Empty title test: {driver.test_empty_title()}")
print(f"Duplicate task test: {driver.test_duplicate_task()}")

## Key Differences

| Aspect | Stub | Driver |
|--------|------|--------|
| Purpose | Replace missing dependencies | Call the module under test |
| Used in | Top-Down integration | Bottom-Up integration |
| Direction | Called by module under test | Calls module under test |
| Complexity | Simple, canned responses | May contain test logic |

---

## When to Use Each

- **Use Stubs when:**
  - Testing higher-level modules first (Top-Down)
  - Dependencies are not yet implemented
  - You need to simulate specific response scenarios
  - You want to isolate the module from external dependencies

- **Use Drivers when:**
  - Testing lower-level modules first (Bottom-Up)
  - Higher-level modules are not yet implemented
  - You need to provide input and verify output
  - You want to test the module in isolation